# **Information**

**Final product, professional practice**

*Santiago Lamas Fresard*

## **Bibliography used**

1. Piazza, A. (2019). A discussion of vintage optimization models in Forest Economics. Forest Science, 66(4), 469–477. https://doi.org/10.1093/forsci/fxz056
2. Pukkala, T., Vauhkonen, J., Korhonen, K. T., & Packalen, T. (2021). Self-learning growth simulator for modelling forest stand dynamics in changing conditions. Forestry an International Journal of Forest Research, 94(3), 333–346. https://doi.org/10.1093/forestry/cpab008
3. Jenkins, J. C., Chojnacky, D. C., Heath, L. S., & Birdsey, R. A. (2003). National-Scale Biomass Estimators for United States Tree Species. Forest Science, 49(1), 12–35. https://doi.org/10.1093/forestscience/49.1.12

# **Inicialization**

Importing of useful libraries.

In [ ]:
import Distributions
import Gurobi
import Ipopt
import JuMP
import LinearAlgebra
import Measures
import Plots
import Random
import Roots
import SDDP

using Distributions
using Gurobi
using Ipopt
using JuMP
using LinearAlgebra
using Measures
using Plots
using Printf
using Random
using Roots
using SDDP

const GRB_ENV = Gurobi.Env();

# **Mitra & Wan's model**

The first part of the work was done on a single-species, multiaged forest model proposed by Mitra & Wan. For a reference on the model with a simpler notation, see Piazza (2019)

## **Estimates of the cost function**

Given the gain function $u(c) = \frac{c^{1-\alpha}}{1-\alpha}$, we seek to find simpler estimates appropiate for the $\texttt{SDDP.jl}$ package.

The first estimate is a quadratic function $\hat u(\cdot)$, chosen so that it minimizes the $L^2([0,T])$ norm of $\hat u(\cdot) - u(\cdot)$, with $T=u^{-1}(UB)$, where $UB$ is an upper bound for the gain function previously chosen. Such quadratic function is of the form $\hat u(c_t) = Ac_t^2 + Bc_t + C$, where the coefficients satisfy the following linear system, which arises from applying a first-order criterion (note that you can differentiate under the integral from the $L^2$ norm):
$$
\begin{equation*}
\begin{cases}
        \frac{T^5}5 A + \frac{T^4}4 B + \frac{T^3}3 C = \frac1{1-\alpha}\frac{T^{4-\alpha}}{4-\alpha}\\
        \frac{T^4}4 A + \frac{T^3}3 B + \frac{T^2}2 C = \frac1{1-\alpha}\frac{T^{3-\alpha}}{3-\alpha}\\
        \frac{T^3}3 A + \frac{T^2}2 B + T C = \frac1{1-\alpha}\frac{T^{2-\alpha}}{2-\alpha}\\
\end{cases}
\end{equation*}
$$

In [ ]:
"""
     params_quad(α:Float64, T:Float64) -> Vector{Float64}

Returns the coefficients of the quadratic function that minimizes 
    the gain function u(c) = \\frac{c^{1-\\alpha}}{1-\\alpha}.

# Arguments
- `α::Float64`: parameter that fixes the gain function.
- `T::Float64`: upper bound for the total biomass extracted.

"""
function params_quad(α, T) 
    A = [T^5/5 T^4/4 T^3/3; T^4/4 T^3/3 T^2/2; T^3/3 T^2/2 T]
    b = [T^(4-α)/(4-α); T^(3-α)/(3-α); T^(2-α)/(2-α)]/(1-α)
    return A^-1*b
end;

An example is given.

In [ ]:
α_ex, T_ex = 0.5, 10000
params_ex = params_quad(α_ex, T_ex)

quad_ex = x -> sum(params_ex.*[x^2, x, 1])
cost_ex = x -> x^(1-α_ex)/((1-α_ex))
l2norm = x -> (quad_ex(x) - cost_ex(x))^2

p1 = plot(
    cost_ex, 
    0.01, 
    T_ex, 
    label="Gain function"
    )
plot!(
    quad_ex, 
    0.01, 
    T_ex, 
    label="Quadratic approximation", 
    title="Comparison")
p2 = plot(
    l2norm, 
    0.01, 
    T_ex, 
    yaxis=:log10, 
    label="Quadratic error", 
    title = "Error")
    
plot(
    p1, p2, 
    layout=(2,1), 
    size=(600,500), 
    plot_title="Quadratic estimate",
    plot_titlevspan=0.15)

Next, two similar methods are proposed and visualized. Both seek to approximate the set $\{(c,y): u(c)\leq y\}$ via an intersection of semiplanes, with the difference between them being the choice of hyperplanes: the first one, nicknamed the *tangent method*, makes each line separating the semiplanes a tangent to the set $\{(c,y): u(c) = y\}$, while the second, nicknamed the *secant method*, does the analogous with secant lines. Both methods need a colection of reference points to draw the lines, which are chosen uniformly either on the X-axis or on the Y-axis. 

Thus, if there are $n$ semiplantes, the separating lines are of the form $\{c_t \mapsto a_i c_t + b_i\}_{i=1}^n$. When choosing the points $\{x_i\}_{i=1}^n$ on the x-axis, or equivalently $\{y_i\}_{i=1}^n$ on the y-axis, the coefficients $\{a_i\}_{i=1}^n$ are defined as follows:

|  | Tangent method | Secant method |
|---|---|---|
| Using $\{x_i\}_{i=1}^n$ | $u'(x_i)$ | $u'(u^{-1}(y_i))$ |
| Using $\{y_i\}_{i=1}^n$ | $\frac{u(x_{i+1}) - u(x_i)}{x_{i+1} - x_i}$ | $\frac{y_{i+1} - y_i}{u^{-1}(y_{i+1}) - u^{-1}(y_i)}$ |

On the other hand, the definition of the coefficients $\{b_i\}_{i=1}^n$ coincide between methods:

|  | Either metod |
|---|---|
| Using $\{x_i\}_{i=1}^n$ | $u(x_i) - a_i\cdot x_i$ |
| Using $\{y_i\}_{i=1}^n$ | $u(u^{-1}(y_i)) - a_i\cdot u^{-1}(y_i)$ |

An example of the lines obtained is shown.

In [ ]:
# Parameters to use in the problem
α = 0.5 # risk aversion parameter for the utility function
UB = 500 # upper bound for the objective function

# Useful functions
obj = x -> x^(1-α)/(1-α) # objective function
deriv_obj = x -> x^-α # derivative of the objective function
inv_obj = y -> ((1-α)*y)^(1/(1-α)) # inverse of the objective function

# Parameters to estimate the objective function
T = inv_obj(UB)
nx, ny = 2500, 200 # number of points for each discretization
dx, dy = T/nx, UB/ny # step sizes for each discretization

# Approximation by a quadratic function
a, b, c = params_quad(α, T)
approx_quad = x -> a*x^2 + b*x + c

# Discretization of the working intervals
d_unif_x = 1:dx:T # uniform on x-axis
d_unif_y = 1:dy:UB # uniform on y-axis

# Image and preimage of the discretizations
f_unif_x = obj.(d_unif_x)    # uniform on x-axis
f_inv_y = inv_obj.(d_unif_y) # uniform on y-axis

# Tangent method, uniform on x-axis
a_tgte_x = deriv_obj.(d_unif_x)             
b_tgte_x = f_unif_x .- a_tgte_x .* d_unif_x
 # Tangent method, uniform on y-axis
a_tgte_y = deriv_obj.(f_inv_y)              
b_tgte_y = d_unif_y .- a_tgte_y .* f_inv_y  
# Secant method, uniform on x-axis
a_sec_x = [(f_unif_x[i+1] - f_unif_x[i])/dx for i in 1:nx-1] 
b_sec_x = f_unif_x[1:nx-1] .- a_sec_x .* d_unif_x[1:nx-1]
# Secant method, uniform on y-axis
a_sec_y = [dy/(f_inv_y[i+1] - f_inv_y[i]) for i in 1:ny-1]   
b_sec_y = d_unif_y[1:ny-1] .- a_sec_y .* f_inv_y[1:ny-1];    

In [ ]:
p1 = plot(
    obj, 
    0.1, 
    T, 
    ylims = (0.1,UB+50), 
    label = "Original gain function",
    xaxis = "Total biomass [Kg]",
    yaxis = "Gain [\$]", 
    title = "Tangent method, points uniform on x-axis"
    )
plot!(
    [x -> a_tgte_x[i]*x + b_tgte_x[i] for i in 10:300:nx], 
    label=false, 
    linestyle=:dot
    )
p2 = plot(
    obj, 
    0.1, 
    T, 
    ylims = (0.1,UB+50), 
    label = "Original gain function",
    xaxis = "Total biomass [Kg]",
    yaxis = "Gain [\$]", 
    title = "Tangent method, points uniform on y-axis"
    )
plot!([x -> a_tgte_y[i]*x + b_tgte_y[i] for i in 10:20:ny], label=false, linestyle=:dot)
p3 = plot(
    obj, 
    0.1, 
    T, 
    ylims = (0.1,UB+50), 
    label = "Original gain function",
    xaxis = "Total biomass [Kg]",
    yaxis = "Gain [\$]", 
    title = "Secant method, point uniform on x-axis"
    )
plot!(
    [x -> a_sec_x[i]*x + b_sec_x[i] for i in 10:300:nx-1], 
    label=false, 
    linestyle=:dot
    )
p4 = plot(
    obj, 
    0.1, 
    T, 
    ylims = (0.1,UB+50), 
    label = "Original gain function",
    xaxis = "Total biomass [Kg]",
    yaxis = "Gain [\$]", 
    title = "Secant method, point uniform on y-axis"
    )
plot!(
    [x -> a_sec_y[i]*x + b_sec_y[i] for i in 10:20:ny-1], 
    label=false, 
    linestyle=:dot
    )
plot(p1, p2, p3, p4, 
    layout = (2,2), 
    size = (1000,800), 
    plot_title = "Estimates of the gain function using lines"
    )

## **Model testing**

We test all the methods previously defined on the Mitra & Wan's model.

### **Important parameters**

In [ ]:
UB = 5000 # upper bound for the objective function
b = 0.975 # discount factor used by Buongiorno
α = 0.5 # risk aversion parameter for the utility function
n = 5 # number of age classes
f = [0.8, 4.2, 16.99, 68.1, 84] # biomass coeffcients
x0 = [1.0, 0.0, 0.0, 0.0, 0.0] # initial state

# Cyclic graph consisting of one node with a probability of (1-b) of exiting the cycle 
graph = SDDP.UnicyclicGraph(b; num_nodes = 1)

# Number of training iterations for the models and number of simulations to compute the bounds
n_train_iterations = 50
n_simulations = 100;

In [ ]:
"""
     build_model(coefs_a:Vector{Float64}, coefs_b:Vector{Float64})

Given two sets of coeffcients, builds a policy graph according to the Mitra & Wan's model,
approximating the gain function with a series of lines, where the first set of coeffcients 
are the slopes and the second set are the y-intercepts.

After this, it procedes to train it and estimates both an upper bound and a confidence interval
for the objective function through a series of simulations.

# Arguments
- `coefs_a::Vector{Float64}`: slopes of the approximating lines.
- `coefs_b::Vector{Float64}`: y-intercepts of the approximating lines.
"""
function build_model(coefs_a, coefs_b)
    n_coefs = length(coefs_a)
    model = SDDP.PolicyGraph(
        graph;
        sense = :Max,
        upper_bound = UB,
        # optimizer = Ipopt.Optimizer,
        optimizer = () -> Gurobi.Optimizer(GRB_ENV),
    ) do sp, t
        # variable: madera 
        @variable(sp, x[s = 1:n] >=0, SDDP.State, initial_value = x0[s]) # state variables, multidimensional.
        @variable(sp, c[1:n] >= 0) # control variables
        @variable(sp, TV >= 0) # Timber volume
        @variable(sp, u >= 0) # slack variable to maximize, forced to be approx. TV^(1-α)/(1-α) 
        @constraints(sp, begin
            c[n] == x[n].in # we harvest everything for the oldest age class
            [s = 1:n-1], x[s+1].out == x[s].in - c[s] # balance equations
            sum(f[s] * c[s] for s in 1:n) == TV # timber volume
            [s = 1:n], x[s].out <= 1 # state space constraints
            sum(x[s].out for s in 1:n) == 1 # x[1].out will be determined by this equation because the other x[s].out are determined by the balance equations
            [s = 1:n_coefs], u <= coefs_a[s]*TV + coefs_b[s] #forces u to be approx. TV^(1-α)/(1-α)
        end)
        @stageobjective(sp, u)
    end
    SDDP.numerical_stability_report(model)
    SDDP.train(model, iteration_limit = n_train_iterations)
    # Simulations to compute bounds. These will have different lengths.
    simulations = SDDP.simulate(model, n_simulations, [:x])
    objective_values = [
        sum(stage[:stage_objective] for stage in sim) for sim in simulations
    ]
    μ, ci = round.(SDDP.confidence_interval(objective_values, 1.96); digits = 2)
    upper_bound = SDDP.calculate_bound(model)
    println("Confidence interval: ", μ, " ± ", ci)
    println("Upper bound: ", round(upper_bound, digits = 2))
end;

### **Quadratic function**

In [ ]:
model0 = SDDP.PolicyGraph(
    graph;
    # maximizamos función de retorno
    sense = :Max,
    upper_bound = UB,
    #optimizer = Ipopt.Optimizer,
    optimizer = () -> Gurobi.Optimizer(GRB_ENV),
) do sp, t
    # variable: madera 
    @variable(sp, x[s = 1:n] >=0, SDDP.State, initial_value = x0[s]) # state variables, multidimensional. Time to figure this out: 3 hours!!!!
    @variable(sp, c[1:n] >= 0) # control variables
    @variable(sp, TV >= 0) # Timber volume
    @constraints(sp, begin
        c[n] == x[n].in # we harvest everything for the oldest age class
        [s = 1:n-1], x[s+1].out == x[s].in - c[s] # balance equations
        sum(f[s] * c[s] for s in 1:n) == TV # timber volume
        [s = 1:n], x[s].out <= 1 # state space constraints
        sum(x[s].out for s in 1:n) == 1 # x[1].out will be determined by this equation because the other x[s].out are determined by the balance equations
    end)
    @stageobjective(sp, approx_quad(TV))
end

println("Aproximación cuadrática:")
SDDP.numerical_stability_report(model0)

SDDP.train(model0, iteration_limit = n_train_iterations)
# Simulations to compute bounds. These will have different lengths.
simulations = SDDP.simulate(model0, n_simulations, [:x])
objective_values = [
    sum(stage[:stage_objective] for stage in sim) for sim in simulations
]
μ, ci = round.(SDDP.confidence_interval(objective_values, 1.96); digits = 2)
upper_bound = SDDP.calculate_bound(model0)
println("Confidence interval: ", μ, " ± ", ci)
println("Upper bound: ", round(upper_bound, digits = 2))

### **Tangent method, points uniform on the x-axis**

In [ ]:
build_model(a_tgte_x, b_tgte_x)

### **Tangent method, points uniform on the y-axis**

In [ ]:
build_model(a_tgte_y, b_tgte_y)

### **Secant method, points uniform on the x-axis**

In [ ]:
build_model(a_sec_x, b_sec_x)

### **Secant method, points uniform on the y-axis**

In [ ]:
build_model(a_sec_y, b_sec_y)

# **Pukkala's model**

The following part of the work was done over a simplification done by Pukkala, Vauhkonen, Korhonen and Packalen. For more information, see Pukkala et al. (2021).

They proposed a model for forest evolution based on three sub-processes: diameter increment ($DI$), survival rates ($S$), and natural ingrowth ($NI$), all measured over five-year periods. Specifically, the functions they propose are of the following form:
- $DI(x) = \exp(f(x))$,
- $S(x) = \frac1{1+\exp(-g(x))}$,
- $NI(x) = \exp(h(x))$,
where the functions $f,\ g,\ h$ are affine functions of a state vector $x$, whose coefficients were subsequently estimated for each species.

We opted to implement an approximation of the Pukkala et al. model, thus studying centering on diameter classes with a range of 5 cm each, evolving every five years. The three previously mentioned functions presented highly complex state vectors, with variables such as temperature, soil quality, and the presence of peatlands. Given these, certain hypotheses were assumed to facilitate subsequent work:
- Mesophilic environment, that is, with moderate humidity and temperature conditions.
- Mineral soil with a high percentage of inorganic material (in contrast to, for example, peatlands).
- The sum of effective temperatures (sum of non-negative temperatures) was set at $TS = 1100$ degree days per year.
- Only one species was studied at a time.
- All trees in the same diameter class shared the same diameter, being equal to the average value.

Thus, the functions became the following:
- $f(x) = a_0 + a_1\sqrt{d} + a_2 d + a_3\log(BA+1) + a_4 \frac{BAL}{\sqrt{d+1}}+ a_5\log(TS)$.
- $g(x) = b_0 + b_1\sqrt{d} + b_2 d + b_3 \frac{BAL}{\sqrt{d+1}}$.
- $h(x) = c_0 + c_1\log(TS) + c_2\sqrt{BA}$.

Here, $BA$ stands as total basal area measured in $\text{m}^2$ per $\text{ha}$., while $BAL$ is the total basal area of trees larger than the one $f$ and $g$ are being applied to.

Using data from the Finnish National Forest Inventory (NFI), the authors determined the parameters for different species types through statistical adjustment based on basal area and the number of trees observed over a five-year period. Specifically, to determine the survival function $S$, *fixed-effects* and *mixed-effects* models were proposed, corresponding to those that assume independence between trees belonging to the same plots and those from different plots, respectively.

The main species used were *Picea abies*, *Pinus sylvestris*, and *Betula pendula*, corresponding to Norway spruce, Scots pine, and Silver birch, respectively. In addition, other broadleaf species were considered, such as *Betula pubescens*, *Populus tremula* and *Alnus incana*, scientific names for the downy birch, aspen and grey alder respectively.

As a first step to an implementation, some paremeters were fixed, as for example an initial condition to study.

In [ ]:
num_classes = 8 # number of diameter classes to study
range_classes = 5 # range of each class
avg_diams = [range_classes*(k-1/2) for k in 1:num_classes] # average diameters of each class
BAs = [pi*d^2/40000 for d in avg_diams] # basal area per tree of each class
X0 = [500, 0, 0, 0, 0, 0, 0, 0]; # initial condition

## **Fixed basal areas**

### **Modeling**

A first implementation of the model followed certain sections of Pukkala et al.'s paper, assuming a constant basal area of ​​$BA = 25$ m² and a basal area for larger trees (than the one studied) of $BAL = 10$ m². Thus, all functions became single-variate, depending only on the diameter of the individual trees they were applied to.

First, a transition matrix between classes was implemented based on the diameter increment function $DI$, where for each class the fraction that remained within the same class after five years was calculated as:
$$
P_{i,i} = \frac{\text{Leb}(DI([5(i-1), 5i]) \cap [5(i-1), 5i])}{5},
$$
that is, the fraction of the interval associated with the class that remained in that interval after applying the $DI$ function ($\text{Leb}(\cdot)$ stands for the Lebesgue measure). Since this function was never greater than $5$, the remaining fraction moved to the next class.

In particular, a dictionary was created, where the transition matrix for each species was saved.

In [ ]:
# Coefficients used for diameter increase
coefs_diam_pine = [-7.1552, 0.4415, -0.0685, -0.2027, -0.1236, 1.1198]
coefs_diam_spruce = [-12.7527, 0.1693, -0.0301, -0.1875, -0.0563, 1.9747]
coefs_diam_broadleaf = [-8.6306, 0.5097, -0.0829, -0.3864, -0.0545,  1.3163]

"""
     diam_fn(d:Float64, coefs:Vector{Float64})-> Float64

Returns DI(d) = exp(f(d)), based on the coefficients.

# Arguments
- `d::Float64`: diameter of the tree the function DI is being applied to.
- `coefs::Vector{Float64}`: coefficients of the function f.
"""
diam_fn = (d, coefs) -> exp(sum(coefs.*[1, sqrt(d), d, log(25+1), 10/sqrt(d+1), log(1100)]))

"""
     tr_diam(fn:Function)-> Matrix{Float64}

Returns the transition matrix P, taking DI = fn.

# Arguments
- `fn:Function`: Diameter-increase function.
"""
function tr_diam(fn)
    tr = zeros(num_classes,num_classes)
    for i in 1:num_classes-1
        # The help function gives the increased diameter for each class, minus the maximum diamter 
        # of said class. Thus, its unique zero (TBP) is the diameter in the class that reaches exactly
        # that maximum diameter. In consequence, all diameters before it remain in the class.
        help = d -> d + fn(d) - range_classes*i
        crit = find_zero(help, range_classes*i)
        frac = (crit - range_classes*(i-1))/range_classes
        tr[i,i] = frac
        tr[i+1,i] = 1 - frac
    end
    # It's asummed that all trees on the larger class remain there.
    tr[end,end] = 1
    return tr
end

# Transition matrices for each species
diam = Dict()
diam["Pine"] = tr_diam(d -> diam_fn(d, coefs_diam_pine))
diam["Spruce"] = tr_diam(d -> diam_fn(d, coefs_diam_spruce))
diam["Birch"] = tr_diam(d -> diam_fn(d, coefs_diam_broadleaf))
diam["Other broadleaf species"] = tr_diam(d -> diam_fn(d, coefs_diam_broadleaf));

Even though the function `tr_diam` could take as input only the coefficients, this definition will become useful in later modifications of the model.

On the other hand, a survival rate was associated with each diameter class, corresponding to evaluating the survival function $S$ at the average value of the interval associated with that class. As on the previous case, a dictionary is created to save the survival rates of each species and effects model.

In [ ]:
# Coefficients used for survival
coefs_survival_pine_mixed = [4.1505, 3.1513, -0.3575, -0.4001]
coefs_survival_spruce_mixed = [9.6649, 1.0157, -0.1577, -0.3244]
coefs_survival_broadleaf_mixed = [3.6655, 1.0650, -0.1509, -0.0326]
coefs_survival_pine_fixed = [1.41223, 1.8852, -0.21317, -0.25637]
coefs_survival_spruce_fixed = [5.01677, 0.36902, -0.07504, -0.2319]
coefs_survival_broadleaf_fixed = [1.60895, 0.71578, -0.08236, -0.13481]

"""
     surviv_fn(d:Float64, coefs:Vector{Float64})-> Float64

Returns S(d) = 1/(1+exp(-g(d)), based on the coefficients.

# Arguments
- `d::Float64`: diameter of the tree the function S is being applied to.
- `coefs::Vector{Float64}`: coefficients of the function g.
"""
surviv_fn = (d, coefs) -> 1/( 1+exp( -sum( coefs.*[1, sqrt(d), d, 10/sqrt(d+1)] ) ) )

"""
     survival_eval(coefs:Vector{Float64})-> Vector{Float64}

Returns the survival rates assigned to each class based on the coefficents.

# Arguments
- `coefs::Vector{Float64}`: coefficients of the function g.
"""
survival_eval = coefs -> [ surviv_fn(d, coefs) for d in avg_diams ]

# Survival rates for each species and each effects model.
surviv = Dict()
surviv["Pine", "fixed"] = survival_eval(coefs_survival_pine_fixed)
surviv["Spruce", "fixed"] = survival_eval(coefs_survival_spruce_fixed)
surviv["Birch", "fixed"] = survival_eval(coefs_survival_broadleaf_fixed)
surviv["Other broadleaf species", "fixed"] = survival_eval(coefs_survival_broadleaf_fixed)
surviv["Pine", "mixed"] = survival_eval(coefs_survival_pine_mixed)
surviv["Spruce", "mixed"] = survival_eval(coefs_survival_spruce_mixed)
surviv["Birch", "mixed"] = survival_eval(coefs_survival_broadleaf_mixed)
surviv["Other broadleaf species", "mixed"] = survival_eval(coefs_survival_broadleaf_mixed);

Finally ingrowth, understood as natural regeneration, was kept constant since the regeneration function $NI$ depends exclusively on the total basal area.

In [ ]:
# Coefficients used for ingrowth
coefs_ingrowth_pine = [-6.6933, 1.9051, -0.5035]
coefs_ingrowth_spruce = [-9.6128, 2.2897, -0.8739]
coefs_ingrowth_birch = [-3.2919, 1.5438, -1.2920]
coefs_ingrowth_broadleaf = [-48.4331,  7.6107, -0.2227]

"""
     ingrowth_fn(d:Float64, coefs:Vector{Float64})-> Float64

Returns NI(d) = exp(h(d)), based on the coefficients.

# Arguments
- `d::Float64`: diameter of the tree the function NI is being applied to.
- `coefs::Vector{Float64}`: coefficients of the function h.
"""
ingrowth_fn = (d, coefs) -> exp(sum(coefs.*[1, log(1100), sqrt(25)]))

# Ingrowth functions for each species.
ingrowth = Dict()
ingrowth["Pine"] = d-> ingrowth_fn(d,coefs_ingrowth_pine)
ingrowth["Spruce"] = d-> ingrowth_fn(d,coefs_ingrowth_spruce)
ingrowth["Birch"] = d-> ingrowth_fn(d,coefs_ingrowth_birch)
ingrowth["Other broadleaf species"] = d-> ingrowth_fn(d,coefs_ingrowth_broadleaf);

To show how these functions behave, they are plotted.

In [ ]:
x1 = 0:0.1:num_classes*range_classes
p1 = plot(
    x1, 
    [d -> diam_fn(d, coefs_diam_pine),
    d -> diam_fn(d, coefs_diam_spruce),
    d -> diam_fn(d, coefs_diam_broadleaf)], 
    xlabel="Diameter [cm]", 
    ylabel="Increase in diameter\n [cm]", 
    left_margin=5mm,
    title="Increase in diameter over 5 years", 
    label=["Pine" "Spruce" "Broadleaf species"], 
    legend=:outerright
    )

x2 = 0.1:0.1:num_classes*range_classes
p2 = plot(
    x2, 
    [d -> surviv_fn(d,coefs_survival_pine_fixed), 
    d -> surviv_fn(d, coefs_survival_spruce_fixed), 
    d -> surviv_fn(d, coefs_survival_broadleaf_fixed), 
    d -> surviv_fn(d, coefs_survival_pine_mixed), 
    d -> surviv_fn(d, coefs_survival_spruce_mixed), 
    d -> surviv_fn(d, coefs_survival_broadleaf_mixed)], 
    xaxis=:log10, 
    xlabel="Diameter [cm]", 
    ylabel="Fraction of surviving\ntrees",
    left_margin=5mm,
    title="Survival of trees over 5 years",
    label=["Pine\n(fixed effects)" "Spruce\n(fixed effects)" "Broadleaf species\n(fixed effects)" "Pine\n(mixed effects)" "Spruce\n(mixed effects)" "Broadleaf species\n(mixed effects)"],
    legend=:outerright
    )

x3 = 0.01:1:num_classes*range_classes
p3 = plot(
    x3, 
    [ingrowth["Pine"], 
    ingrowth["Spruce"], 
    ingrowth["Birch"], 
    ingrowth["Other broadleaf species"]],
    xaxis=:log10, 
    xlabel="Diameter [cm]", 
    ylabel="Number of trees \nborn per ha.",
    bottom_margin=5mm, 
    title="Ingrowth over 5 years", 
    label=["Pine" "Spruce" "Birch" "Other broadleaf species"], 
    legend=:outerright
    )
plot(
    p1, p2, p3, 
    layout=(2,2), 
    size=(1000,600), 
    plot_title="Functions of the simplified model: BA and BAL fixed", 
    plot_titlevspan=0.08
    )

### **Simulations**

With the model defined, simulations where ran. Thus, every five years, the state vector had the survival rates applied to each class, after which they evolved according to the transition matrix, adding a number of trees to the first class given by the ingrowth function.

With this end, three functions were created with the purpose of running and plotting the simulations.

In [ ]:
"""
     indiv_tray(num_steps:Int64, initial_config:Vector{Float64}, 
     diam_matrix:Matrix{Float64}, surv_rates:Vector{Float64}, 
     ingrowth_eval:Function)->Matrix{Float64}

Returns a simulation of the model according to the given arguments.

# Arguments
- `num_steps:Int64`: number of steps the simulation takes.
- `initial_config:Vector{Float64}`: initial configuration of the forest.
- `diam_matrix:Matrix{Float64}`: transition matrix for the diameter classes
- `surv_rates:Vector{Float64}`: survival rates for each diameter class.
- `ingrowth_eval:Function`: ingrowth function.
"""
function indiv_tray(num_steps, initial_config, diam_matrix, surv_rates, ingrowth_eval)
    X = Array{Float64}(undef, num_steps, num_classes)
    X[1,:] = initial_config
    for i in 2:num_steps
        X[i,:] = diam_matrix * (X[i-1,:] .* surv_rates)
        X[i,1] += ingrowth_eval(0)
    end
    return X
end;

In [ ]:
# Legends for plotting
legends_pie = ["[0,5] cm", "[5,10] cm", "[10,15] cm", "[15,20] cm", "[20,25] cm", "[25,30] cm",
     "[30,35] cm", "[35,40] cm", "[40,45] cm", "[45,50] cm"]

"""
     create_pie(data:Vector{Float64}, effects:String)

Returns a pie chart of the data with a title that states the effects model.

# Arguments
- `data:Vector{Float64}`: data plotted on the pie chart.
- `effects:String`: effects model to be shown on the title.
"""
function create_pie(data, effects)
    pie_chart = pie(
        legends_pie, 
        data, 
        title = "Final distribution ("*effects*" effects)", 
        l = 0.5, 
        legend = :outerright
        )
    datapct = [@sprintf("%.1f%%", x/sum(data)*100) for x in data]
    θ = (cumsum(data) - data/2) .* 360/sum(data)
    scθ = sincosd.(θ)
    for (s, sci) in zip(datapct, scθ)
        annotate!(1.4*sci[2], 1.4*sci[1], text(s, 6, :black))
    end
    return pie_chart
end;

In [ ]:
# Legends for plotting
legends = ["[0,5] cm" "[5,10] cm" "[10,15] cm" "[15,20] cm" "[20,25] cm" "[25,30] cm" "[30,35] cm" "[35,40] cm" "[40,45] cm" "[45,50] cm"]

"""
     simulate_and_plot(species:String, num_steps:Int64, initial_config:Vector{Float64})

Simulates the model for the chosen species both on the fixed-effects and mixed-effects cases,
plotting both the evolution of the distribution on diameter classes and a pie chart with the
final distribution.

# Arguments
- `species:String`: species modeled.
- `num_steps:Int64`: number of steps the simulation is ran, each one corresponding to five years.
- `initial_config:Vector{Float64}`: initial configuration of the diameter classes.  
"""
function simulate_and_plot(species, num_steps, initial_config)
    # Both effects models are ran using the transition matrix, survival rates and ingrowth function of the species.
    X_fixed = indiv_tray(
        num_steps+1, 
        initial_config, 
        diam[species], 
        surviv[species, "fixed"], 
        ingrowth[species]
        )
    X_mixed = indiv_tray(
        num_steps+1, 
        initial_config, 
        diam[species], 
        surviv[species, "mixed"], 
        ingrowth[species]
        )
    # Evolution of the number of trees
    print("Initial size of the forest: ");println(sum(X_fixed[1,:]))
    print("Final size of the forest (fixed effects): ");println(sum(X_fixed[end,:]))
    print("Final size of the forest (mixed effects): ");println(sum(X_mixed[end,:]))
    pf1 = plot(
        0:5:num_steps*5, 
        X_fixed .+ 1, 
        yaxis=:log10, 
        label = legends, 
        legend = :outerright,
        title = "Evolution (fixed effects)", 
        left_margin=5mm,
        xlabel = "Time [years]", 
        ylabel = "Number of trees + 1"
        )
    pf2 = create_pie(
        X_fixed[end,:], 
        "fixed"
        )
    pm1 = plot(
        0:5:num_steps*5, 
        X_mixed .+ 1, 
        yaxis=:log10, 
        label = legends, 
        legend = :outerright,
        title = "Evolution (mixed effects)", 
        xlabel = "Time [years]", 
        ylabel = "Number of trees + 1"
        )
    pm2 = create_pie(
        X_mixed[end,:], 
        "mixed"
        )
    plot(
        pf1, pm1, pf2, pm2, 
        layout=(2,2), 
        size=(800,700), 
        plot_title=species*"'s models"
        )
end;

#### **Pine**

In [ ]:
simulate_and_plot("Pine",100,X0)

#### **Spruce**

In [ ]:
simulate_and_plot("Spruce",100,X0)

#### **Birch**

In [ ]:
simulate_and_plot("Birch",100,X0)

#### **Other broadleaf species**

In [ ]:
simulate_and_plot("Other broadleaf species",100,X0)

## **Non-fixed basal areas**

### **Modeling**

The second implementation of the model discarded the assumptions of constant ​$BA$ and $BAL$. In particular, due to the previous set of hypotheses, for a tree in a given class, $BAL$ was given by the basal area of all superior classes.

Thus, all previously defined functions had to account for this addition. This implied that the transition matrices and survival rates were dependent on the whole state of the forest.

In [ ]:
"""
     vars_diam_BA(d:Float64, X:Vector{Float64})->Vector{Float64}

Returns the set of variables of the diameter increase function f, particularly depending
on the diameter of the studied tree and the whole forest.

# Arguments
- `d::Float64`: diameter of the tree the function DI is being applied to.
- `X::Vector{Float64}`: vector state of the current forest.
"""
function vars_diam_BA(d, X)
    help_BAL = zeros(num_classes)
    BA = sum(BAs.*X)
    num = floor(Int, d/5) + 1
    help_BAL[num:num_classes] .= 1
    BAL = sum(BAs.*X.*help_BAL)
    return [1, sqrt(d), d, log(BA+1), BAL/sqrt(d+1), log(1100)]
end

"""
     diameter_BA(d:Float64, X:Vector{Float64}, coefs:Vector{Float64})-> Matrix{Float64}

Returns the transition matrix P, taking DI = fn.

# Arguments
- `d::Float64`: diameter of the tree the function DI is being applied to.
- `X::Vector{Float64}`: vector state of the current forest.
- `coefs::Vector{Float64}`: coefficients of the diameter increase function f.
"""
diameter_BA = (d, X, coefs) -> exp(sum(coefs.*vars_diam_BA(d,X)))

# Transition matrix functions for each species based on the state of the forest.
diam_BA = Dict()
diam_BA["Pine"] = X -> tr_diam(d -> diameter_BA(d, X, coefs_diam_pine))
diam_BA["Spruce"] = X -> tr_diam(d -> diameter_BA(d, X, coefs_diam_spruce))
diam_BA["Birch"] = X -> tr_diam(d -> diameter_BA(d, X, coefs_diam_broadleaf))
diam_BA["Other broadleaf species"] = X -> tr_diam(d -> diameter_BA(d, X, coefs_diam_broadleaf));

In [ ]:
"""
     vars_survival_BA(d:Float64, X:Vector{Float64})->Vector{Float64}

Returns the set of variables of the survival function g, particularly depending
on the diameter of the studied tree and the whole forest.

# Arguments
- `d::Float64`: diameter of the tree the function DI is being applied to.
- `X::Vector{Float64}`: vector state of the current forest.
"""
function vars_survival_BA(d, X)
    help_BAL = zeros(num_classes)
    num = floor(Int, d/5) + 1
    help_BAL[num:num_classes] .= 1
    BAL = sum(BAs.*X.*help_BAL)
    return [1, sqrt(d), d, BAL/sqrt(d+1)]    
end

"""
     survival_BA(X:Vector{Float64}, coefs:Vector{Float64})-> Vector{Float64}

Returns the survival rates assigned to each class based on the current state of the forest
and the coefficents.

# Arguments
- `X::Vector{Float64}`: vector state of the current forest.
- `coefs::Vector{Float64}`: coefficients of the survival function g.
"""
survival_BA = (X, coefs) -> [1/(1+exp(-sum(coefs.*vars_survival_BA(d, X)))) for d in avg_diams]

# Survival rates functions for each species and effects model based on the state of the forest.
surviv_BA = Dict()
surviv_BA["Pine", "fixed"] = X -> survival_BA(X, coefs_survival_pine_fixed)
surviv_BA["Spruce", "fixed"] = X -> survival_BA(X, coefs_survival_spruce_fixed)
surviv_BA["Birch", "fixed"] = X -> survival_BA(X, coefs_survival_broadleaf_fixed)
surviv_BA["Other broadleaf species", "fixed"] = X -> survival_BA(X, coefs_survival_broadleaf_fixed)
surviv_BA["Pine", "mixed"] = X -> survival_BA(X, coefs_survival_pine_mixed)
surviv_BA["Spruce", "mixed"] = X -> survival_BA(X, coefs_survival_spruce_mixed)
surviv_BA["Birch", "mixed"] = X -> survival_BA(X, coefs_survival_broadleaf_mixed)
surviv_BA["Other broadleaf species", "mixed"] = X -> survival_BA(X, coefs_survival_broadleaf_mixed);

In [ ]:
"""
     vars_ingrowth_BA(X:Vector{Float64})->Vector{Float64}

Returns the set of variables of the function h, particularly depending
on the whole forest.

# Arguments
- `X::Vector{Float64}`: vector state of the current forest.
"""
vars_ingrowth_BA = X -> [1, log(1100), sqrt(sum(BAs.*X))]  

"""
     ingrowth_BA_coefs(X:Vector{Float64}, coefs:Vector{Float64})-> Float64

Returns the number of ingrown trees based on the current state of the forest and the coefficents.

# Arguments
- `X::Vector{Float64}`: vector state of the current forest.
- `coefs::Vector{Float64}`: coefficients of the ingrowth function h.
"""
ingrowth_BA_coefs = (X, coefs) -> exp(sum(coefs.*vars_ingrowth_BA(X)))

# Ingrowth for each species and effects model based on the state of the forest.
ingrowth_BA = Dict()
ingrowth_BA["Pine"] = X -> ingrowth_BA_coefs(X, coefs_ingrowth_pine)
ingrowth_BA["Spruce"] = X -> ingrowth_BA_coefs(X, coefs_ingrowth_spruce)
ingrowth_BA["Birch"] = X -> ingrowth_BA_coefs(X, coefs_ingrowth_broadleaf)
ingrowth_BA["Other broadleaf species"] = X -> ingrowth_BA_coefs(X, coefs_ingrowth_broadleaf);

As the ingrowth function depends explicitly on the basal area $BA$, we plot then to check for it's behaviour, also givin linear approximations for them.

In [ ]:
"""
     vars_ingrowth_BA_fn(BA:Float64)->Vector{Float64}

Returns the set of variables of the function h, depending on the total basal area.

# Arguments
- `BA::Float64`: total basal area of the forest.
"""
vars_ingrowth_BA_fn = BA -> [1, log(1100), sqrt(BA)]  

"""
     ingrowth_BA_fn(BA:Float64)-> Float64

Returns the number of ingrown trees based on the total basal area and the coefficients.

# Arguments
- `BA::Float64`: total basal area of the forest.
- `coefs::Vector{Float64}`: coefficients of the ingrowth function h.
"""
ingrowth_BA_fn = (BA, coefs) -> exp(sum(coefs.*vars_ingrowth_BA_fn(BA)))

"""
     ingrowth_aprox(BA:Float64, x1:Float64, x2:Float64, coefs_ingrowth:Vector{Float64}))-> Float64

Approximates the ingrowth function with a linear function depending on the basal area and the coefficientes
obtained from interpolating between two chosen points.

# Arguments
- `BA::Float64`: total basal area of the forest.
- `x1::Float64`: first point used for the linear interpolation.
- `x2::Float64`: first point used for the linear interpolation.
- `coefs::Vector{Float64}`: coefficients of the ingrowth function h.
"""
function ingrowth_aprox(BA, x1, x2, coefs_ingrowth)
    y1, y2 = ingrowth_BA_fn(x1,coefs_ingrowth), ingrowth_BA_fn(x2,coefs_ingrowth)
    return (y2-y1)*(BA-x1)/(x2-x1) + y1
end

# Interpolation points for the different species.
x1_pine, x2_pine = 5, 65
x1_spruce, x2_spruce = 5, 45
x1_birch, x2_birch = 5, 45
x1_broadleaf, x2_broadleaf = 5, 45

# Approximated ingrowth for each species and effects model based on the state of the forest.
ingrowth_BA_aprox = Dict()
ingrowth_BA_aprox["Pine"] = BA -> ingrowth_aprox(BA, x1_pine, x2_pine, coefs_ingrowth_pine)
ingrowth_BA_aprox["Spruce"] = BA -> ingrowth_aprox(BA, x1_spruce, x2_spruce, coefs_ingrowth_spruce)
ingrowth_BA_aprox["Birch"] = BA -> ingrowth_aprox(BA, x1_birch, x2_birch, coefs_ingrowth_birch)
ingrowth_BA_aprox["Other broadleaf species"] = BA -> ingrowth_aprox(BA, x1_broadleaf, x2_broadleaf, coefs_ingrowth_broadleaf);

In [ ]:
# Plots for all ingrowth functions, real and approximations.
p1 = plot(
    [BA -> ingrowth_BA_fn(BA,coefs_ingrowth_pine), 
    ingrowth_BA_aprox["Pine"]], 
    0, 
    x2_pine + 5, 
    xlabel="Total basal area [m^2/ha]", 
    ylabel="Trees born per ha.",
    title="Pine", 
    label=["Real" "Aproximation"],
    left_margin=5mm, bottom_margin=5mm
    )
p2 = plot(
    [BA -> ingrowth_BA_fn(BA,coefs_ingrowth_spruce), 
    ingrowth_BA_aprox["Spruce"]], 
    0, 
    x2_spruce + 5,
    xlabel="Total basal area [m^2/ha]", 
    ylabel="Trees born per ha.",
    title="Spruce", 
    label=["Real" "Aproximation"],
    bottom_margin=5mm
    )
p3 = plot(
    [BA -> ingrowth_BA_fn(BA,coefs_ingrowth_birch), 
    ingrowth_BA_aprox["Birch"]], 
    0, 
    x2_birch + 5,
    xlabel="Total basal area [m^2/ha]", 
    ylabel="Trees born per ha.",
    title="Birch", 
    label=["Real" "Aproximation"],
    left_margin=5mm, bottom_margin=5mm
    )
p4 = plot(
    [BA -> ingrowth_BA_fn(BA,coefs_ingrowth_broadleaf), 
    ingrowth_BA_aprox["Other broadleaf species"]], 
    0, 
    x2_broadleaf + 5,
    xlabel="Total basal area [m^2/ha]", 
    ylabel="Trees born per ha.",
    title="Other broadleaf species", 
    label=["Real" "Aproximation"],
    bottom_margin=5mm
    )
    
plot(
    p1, p2, p3, p4, 
    layout=(2,2), 
    size=(600,700), 
    plot_title="Ingrowth as a function of basal area"
    )

### **Simulations**

As in the previous case, the new model is simulated and plotted for each species.

In [ ]:
"""
     traj_BA(num_steps:Int64, initial_config:Vector{Float64}, 
     diam_matrix:Matrix{Float64}, surviv:Vector{Float64}, 
     ingrowth:Function)->Matrix{Float64}

Returns a simulation of the basal-area-dependent model according to the given arguments.

# Arguments
- `num_steps:Int64`: number of steps the simulation takes.
- `initial_config:Vector{Float64}`: initial configuration of the forest.
- `diam_matrix:Function`: function giving the transition matrix for the diameter classes
- `surviv:Function`: function returning survival rates for each diameter class.
- `ingrowth:Function`: ingrowth function.
"""
function traj_BA(num_steps, initial_config, diam_matrix, surviv, ingrowth)
    X = Array{Float64}(undef, num_steps, num_classes)
    X[1,:] = initial_config
    for i in 2:num_steps
        Z = X[i-1,:]
        X[i,:] = diam_matrix(X[i-1,:] .* surviv(Z)) * X[i-1,:]
        X[i,1] += ingrowth(Z)
    end
    return X
end;

In [ ]:
"""
     simulate_and_plot_BA(species:String, num_steps:Int64, initial_config:Vector{Float64})

Simulates the base-dependent model for the chosen species both on the fixed-effects and 
mixed-effects cases, plotting both the evolution of the distribution on diameter classes 
and a pie chart with the final distribution.

# Arguments
- `species:String`: species modeled.
- `num_steps:Int64`: number of steps the simulation is ran, each one corresponding to five years.
- `initial_config:Vector{Float64}`: initial configuration of the diameter classes.  
"""
function simulate_and_plot_BA(species, num_steps, initial_config)
    X_fixed = traj_BA(
        num_steps+1, 
        initial_config, 
        diam_BA[species], 
        surviv_BA[species, "fixed"], 
        ingrowth_BA[species]
        )
    X_mixed = traj_BA(
        num_steps+1, 
        initial_config, 
        diam_BA[species], 
        surviv_BA[species, "mixed"], 
        ingrowth_BA[species]
        )
    print("Initial size of the forest: ");println(sum(X_fixed[1,:]))
    print("Final size of the forest (fixed effects): ");println(sum(X_fixed[end,:]))
    print("Final size of the forest (mixed effects): ");println(sum(X_mixed[end,:]))
    pf1 = plot(
        0:5:num_steps*5, 
        X_fixed .+ 1, 
        yaxis=:log10, 
        label = legends, 
        legend = :outerright,
        title = "Evolution (fixed effects)", 
        left_margin=5mm,
        xlabel = "Time [years]", 
        ylabel = "Number of trees + 1"
        )
    pf2 = create_pie(
        X_fixed[end,:], 
        "fixed"
        )
    pm1 = plot(
        0:5:num_steps*5, 
        X_mixed .+ 1, 
        yaxis=:log10, 
        label = legends, 
        legend = :outerright,
        title = "Evolution (mixed effects)", 
        xlabel = "Time [years]", 
        ylabel = "Number of trees + 1"
        )
    pm2 = create_pie(
        X_mixed[end,:], 
        "mixed"
        )
    plot(
        pf1, pm1, pf2, pm2, 
        layout=(2,2), 
        size=(800,700), 
        plot_title=species*"'s models")
end;

In [ ]:
simulate_and_plot_BA("Pine", 100, X0)

In [ ]:
simulate_and_plot_BA("Spruce", 100, X0)

In [ ]:
simulate_and_plot_BA("Birch", 100, X0)

In [ ]:
simulate_and_plot_BA("Other broadleaf species", 100, X0)

# **Biomass: Jenkins's model**

This section of work was inspired by a model of tree biomass done by Jenkins, Chojnacky, Heath and Birdsey. For more information, see Jenkins et al. (2003)

Their model proposes that the above-ground biomass (the dry weight in kg of organic matter) of a tree, as a function of diameter, is given by the following expression:
$$
bm(d) = exp(\beta_0 +  \beta_1\log d),
$$
with coefficients $\beta_0$ and $\beta_1$ to be determined.

After a massive compilation of models proposed up to that point, the authors generated pseudodata from each of them. They then determined the coefficients for different genera of tree species common in the United States using logarithmic regressions, obtaining quasi-quadratic functions as a function of diameter. These genera included *Picea*, *Pinus*, and a group of broadleaf species.

## **Implementation**

To use said biomass functions, the biomass of each diameter class was fixed as the biomass assigned to the average diameter of the class. As before, these values are stored on a Dictionary.

In [ ]:
# Coefficients used for biomass
coefs_biomass_pine = [-2.5356, 2.4349]
coefs_biomass_spruce = [-2.0773, 2.3323]
coefs_biomass_broadleaf = [-1.9123, 2.3651]

"""
     biomass_fn(d:Float64, coefs:Vector{Float64})-> Float64

Returns bm(d), based on the coefficients.

# Arguments
- `d::Float64`: diameter of the tree the function bm is being applied to.
- `coefs::Vector{Float64}`: coefficients of the function bm.
"""
biomass_fn = (d, coefs) -> exp(sum(coefs.*[1, log(d)]))

"""
     biomass_eval(coefs:Vector{Float64})-> Vector{Float64}

Returns the biomass values assigned to each class based on the coefficents.

# Arguments
- `coefs::Vector{Float64}`: coefficients of the function bm.
"""
biomass_eval = coefs -> [biomass_fn(d, coefs) for d in avg_diams]

# Biomass values for each species.
biomass = Dict()
biomass["Pine"] = biomass_eval(coefs_biomass_pine)
biomass["Spruce"] = biomass_eval(coefs_biomass_spruce)
biomass["Birch"] = biomass_eval(coefs_biomass_broadleaf)
biomass["Other broadleaf species"] = biomass_eval(coefs_biomass_broadleaf);

To compare between the functions, they are plotted.

In [ ]:
p1 = plot(
    d -> biomass_fn(d,coefs_biomass_pine), 
    0, 
    40, 
    legend=false,
    xaxis="Diameter [cm]", 
    yaxis="Biomass [kg]", 
    left_margin=5mm,
    bottom_margin=5mm, 
    title="Pine"
    )
p2 = plot(
    d -> biomass_fn(d,coefs_biomass_spruce), 
    0, 
    40, 
    legend=false,
    xaxis="Diameter [cm]", 
    yaxis="Biomass [kg]", 
    left_margin=5mm,
    bottom_margin=5mm, 
    title="Spruce"
    )
p3 = plot(
    d -> biomass_fn(d,coefs_biomass_broadleaf), 
    0, 
    40, 
    legend=false,
    xaxis="Diameter [cm]", 
    yaxis="Biomass [kg]", 
    left_margin=5mm,
    bottom_margin=5mm, 
    title="Broadleaf species"
    )

plot(
    p1,p2,p3, 
    size=(1000,350), 
    layout=(1,3), 
    plot_title="Biomass as a function of diameter",
    plot_titlevspan=0.1
    )

# **Optimization**

To finally implement the models on the $\texttt{SDDP.jl}$ library, first some initial pareters are set in place.

In [ ]:
x0 = [60 for i in 1:num_classes] # initial condition
UB = 80000 # upper bound for the gain function
test_BA = 25; # test basal area for the model that doesn't depend on it.

## **Independent of basal area**

The first model supposes that the ingrowth function is constant for every species, given by its value evaluated on a chosen basal area.

In [ ]:
"""
     indep_model(species:String, effect:String)->Tuple{Matrix{Float64}, Matrix{Float64}, Vector{Float64}}

Builds a policy graph based on Pukkala's model under the assumption of fixed basal areas. After this, 
it trains the policy graph and estimates both an upper bound and a confidence interval
for the objective function through a series of simulations. Furthermore, it returns the evolution of the state vector,
the control vector (chopped down trees) and the basal area of the longest simulation.

# Arguments
- `species::String`: species to be simulated.
- `effect::String`: model for the survival paremeters to be simulated.
"""
function indep_model(species, effect)
    model = SDDP.PolicyGraph(
        graph;
        sense = :Max,
        upper_bound = UB,
        # optimizer = Ipopt.Optimizer,
        optimizer = () -> Gurobi.Optimizer(GRB_ENV),
    ) do sp, t
        # variable: madera 
        @variable(sp, x[s = 1:num_classes] >=0, SDDP.State, initial_value = x0[s]) # state variables, multidimensional.
        @variable(sp, c[1:num_classes] >= 0) # control variables
        @variable(sp, BA >= 0) # basal area
        @variable(sp, TV >= 0) # Timber volume
        @variable(sp, u >= 0) # slack variable to maximize, forced to be approx. TV^(1-α)/(1-α) 
        @constraints(sp, begin
            [s = 2:num_classes], x[s].out == sum(diam[species][s,t]*((x[t].in - c[t])*surviv[species,effect][t]) for t in 1:num_classes) # balance equations
            sum(BAs[s] * x[s].out for s in 2:num_classes) == BA
            x[1].out == diam[species][1,1]*(x[1].in - c[1])*surviv[species, effect][1] + ingrowth_BA[species](test_BA)
            sum(biomass[species][s] * c[s] for s in 1:num_classes) == TV # timber volume
            [s = 1:ny], u <= a_tgte_y[s]*TV + b_tgte_y[s] #forces u to be approx. TV^(1-α)/(1-α)
        end)
        @stageobjective(sp, u)
    end 
    SDDP.numerical_stability_report(model)
    SDDP.train(model, iteration_limit = n_train_iterations)
    # Simulations to compute bounds. These will have different lengths and have information on the variables x, c and BA
    simulations = SDDP.simulate(model, n_simulations, [:x, :c, :BA])
    # We extract the largest simulation's index and length
    max_len, argmax_len = maximum(length.(simulations)), argmax(length.(simulations))
    # simulations[x][y][:z] is the z variable at the y-th step on the x-th simulation
    Y = reduce(hcat, [[simulations[argmax_len][t][:x][s].in for t in 1:max_len] for s in 1: num_classes])
    C = reduce(hcat, [[simulations[argmax_len][t][:c][s] for t in 1:max_len] for s in 1: num_classes])
    BA = [simulations[argmax_len][t][:BA] for t in 1:max_len]
    objective_values = [
        sum(stage[:stage_objective] for stage in sim) for sim in simulations
    ]
    μ, ci = round.(SDDP.confidence_interval(objective_values, 1.96); digits = 2)
    upper_bound = SDDP.calculate_bound(model)
    println("Confidence interval: ", μ, " ± ", ci)
    println("Upper bound: ", round(upper_bound, digits = 2))
    return (Y, C, BA)
end;

In [ ]:
function plot_trajectories(model_fixed, model_mixed, species)
    Y_fixed, C_fixed, BA_fixed = model_fixed
    Y_mixed, C_mixed, BA_mixed = model_mixed
    print("Initial size of the forest: ");println(sum(Y_fixed[1,:]))
    print("Final size of the forest (fixed effects): ");println(sum(Y_fixed[end,:]))
    print("Final size of the forest (mixed effects): ");println(sum(Y_mixed[end,:]))
    print("Maximum basal area (fixed effects): ");println(maximum(BA_fixed))
    print("Maximum basal area (mixed effects): ");println(maximum(BA_mixed))
    pf1 = plot(
        Y_fixed .+ 1, 
        yaxis=:log10, 
        label = legends, 
        left_margin=5mm, 
        bottom_margin=5mm,
        legend = :outerright,
        title = "Evolution (fixed effects)", 
        xlabel = "Time [Years]", 
        ylabel = "Number of trees + 1"
        )
    pf2 = create_pie(
        Y_fixed[end,:], 
        "fixed"
        )
    pf3 = plot(
        BA_fixed, 
        label="Basal area",
        legend = :outerright, 
        title="Basal area (fixed effects)", 
        xlabel="Time [Years]", 
        ylabel = "Total area [m^2/ha]"
        )
    pf4 = plot(
        C_fixed .+ 1, 
        yaxis=:log10, 
        label = legends,
        legend = :outerright, 
        title = "Optimal policy (fixed effects)", 
        left_margin=5mm,
        xlabel = "Time [Years]", 
        ylabel = "Number of trees + 1"
        )

    pm1 = plot(
        Y_mixed .+ 1, 
        yaxis=:log10, 
        label = legends, 
        bottom_margin=5mm,
        legend = :outerright,
        title = "Evolution (mixed effects)", 
        xlabel = "Time [Years]", 
        ylabel = "Number of trees + 1"
        )
    pm2 = create_pie(
        Y_mixed[end,:], 
        "mixed"
        )
    pm3 = plot(
        BA_mixed, 
        label="Basal area",
        legend = :outerright, 
        title="Basal area (mixed effects)", 
        xlabel="Time [Years]", 
        ylabel = "Total area [m^2/ha]"
        )
    pm4 = plot(
        C_mixed .+ 1, 
        yaxis=:log10, 
        label = legends,
        legend = :outerright, 
        title = "Optimal policy (mixed effects)", 
        left_margin=5mm,
        xlabel = "Time [Years]", 
        ylabel = "Number of trees + 1"
        )

    plot(
        pf1, pm1, pf2, pm2, pf3, pm3, pf4, pm4, 
        layout=(4,2), 
        size=(850,1200), 
        plot_title=species*"'s model"
        )
end;

### **Pine**

In [ ]:
model_pine_fixed1 = indep_model("Pine","fixed");

In [ ]:
model_pine_mixed1 = indep_model("Pine","mixed");

In [ ]:
plot_trajectories(model_pine_fixed1, model_pine_mixed1, "Pine")

### **Spruce**

In [ ]:
model_spruce_fixed1 = indep_model("Spruce","fixed");

In [ ]:
model_spruce_mixed1 = indep_model("Spruce","mixed");

In [ ]:
plot_trajectories(model_spruce_fixed1, model_spruce_mixed1, "Spruce")

### **Birch**

In [ ]:
model_birch_fixed1 = indep_model("Birch","fixed");

In [ ]:
model_birch_mixed1 = indep_model("Birch","mixed");

In [ ]:
plot_trajectories(model_birch_fixed1, model_birch_mixed1, "Birch")

### **Broadleaf**

In [ ]:
model_broadleaf_fixed1 = indep_model("Other broadleaf species","fixed");

In [ ]:
model_broadleaf_mixed1 = indep_model("Other broadleaf species","mixed");

In [ ]:
plot_trajectories(model_broadleaf_fixed1, model_broadleaf_mixed1, "Broadleaf")

## **Linear approximation**

Now we use the approximation of the ingrowth function, as a function of total basal area, via the previous linear interpolation.

In [ ]:
"""
     dep_model(species:String, effect:String, ingrowth_fn:Function)->SDDP.PolicyGraph{Int64}

Returns a policy graph based on Pukkala's model under the assumption of a non-constant ingrowth function.

# Arguments
- `species::String`: species to be simulated.
- `effect::String`: model for the survival paremeters to be simulated.
- `ingrowth_fn::Function`: ingrowth function used in the model.
"""
function dep_model(species, effect, ingrowth_fn)
    model = SDDP.PolicyGraph(
        graph;
        sense = :Max,
        upper_bound = UB,
        # optimizer = Ipopt.Optimizer,
        optimizer = () -> Gurobi.Optimizer(GRB_ENV),
    ) do sp, t
        # variable: madera 
        @variable(sp, x[s = 1:num_classes] >=0, SDDP.State, initial_value = x0[s]) # state variables, multidimensional.
        @variable(sp, c[1:num_classes] >= 0) # control variables
        @variable(sp, BA >= 0) # basal area
        @variable(sp, TV >= 0) # Timber volume
        @variable(sp, u >= 0) # slack variable to maximize, forced to be approx. TV^(1-α)/(1-α) 
        @constraints(sp, begin
            [s = 2:num_classes], x[s].out == sum(diam[species][s,t]*((x[t].in - c[t])*surviv[species,effect][t]) for t in 1:num_classes) # balance equations
            sum(BAs[s] * x[s].out for s in 2:num_classes) == BA
            x[1].out == diam[species][1,1]*(x[1].in - c[1])*surviv[species, effect][1] + ingrowth_fn[species](BA)
            sum(biomass[species][s] * c[s] for s in 1:num_classes) == TV # timber volume
            [s = 1:ny], u <= a_tgte_y[s]*TV + b_tgte_y[s] #forces u to be approx. TV^(1-α)/(1-α)
        end)
        @stageobjective(sp, u)
    end 
    return model
end

"""
     train(model:SDDP.PolicyGraph{Int64})->SDDP.PolicyGraph{Int64}

Returns a trained model.

# Arguments
- `model::SDDP.PolicyGraph{Int64}`: model to be trained.

"""
function train(model)
    SDDP.numerical_stability_report(model)
    SDDP.train(model, iteration_limit = n_train_iterations)
    return model
end

"""
     simulate(model:SDDP.PolicyGraph{Int64})->Tuple{Matrix{Float64}, Matrix{Float64}, Vector{Float64}}

Estimates both an upper bound and a confidence interval for the objective function through a 
series of simulations. Furthermore, it returns the evolution of the state vector, the control vector 
(chopped down trees) and the basal area of the longest simulation.

# Arguments
- `model::SDDP.PolicyGraph{Int64}`: model to be simulated.
"""
function simulate(model)
    simulations = SDDP.simulate(model, 200, [:x, :c, :BA])
    max_len, argmax_len = maximum(length.(simulations)), argmax(length.(simulations))
    Y = reduce(hcat, [[simulations[argmax_len][t][:x][s].in for t in 1:max_len] for s in 1: num_classes])
    C = reduce(hcat, [[simulations[argmax_len][t][:c][s] for t in 1:max_len] for s in 1: num_classes])
    BA = [simulations[argmax_len][t][:BA] for t in 1:max_len]
    objective_values = [
        sum(stage[:stage_objective] for stage in sim) for sim in simulations
    ]
    μ, ci = round.(SDDP.confidence_interval(objective_values, 1.96); digits = 2)
    upper_bound = SDDP.calculate_bound(model)
    println("Confidence interval: ", μ, " ± ", ci)
    println("Upper bound: ", round(upper_bound, digits = 2))
    return (Y, C, BA)
end

"""
     dep_model_all(species:String, effect:String, ingrowth_fn:Function)->Tuple{Matrix{Float64}, Matrix{Float64}, Vector{Float64}}

Builds a policy graph based on Pukkala's model under the assumption of non-fixed basal areas. After this, 
it trains the policy graph and estimates both an upper bound and a confidence interval
for the objective function through a series of simulations. Furthermore, it returns the evolution of the state vector,
the control vector (chopped down trees) and the basal area of the longest simulation.

# Arguments
- `species::String`: species to be simulated.
- `effect::String`: model for the survival paremeters to be simulated.
- `ingrowth_fn::Function`: ingrowth function used in the model.
"""
dep_model_all = (species, effect, ingrowth_fn) -> simulate(train(dep_model(species, effect, ingrowth_fn)));

Although it seems a bit unintuitive to have all these separete definitions, this will come in handy later.

### **Pine**

In [ ]:
model_pine_fixed2 = dep_model_all("Pine","fixed",ingrowth_BA_aprox);

In [ ]:
model_pine_mixed2 = dep_model_all("Pine","mixed",ingrowth_BA_aprox);

In [ ]:
plot_trajectories(model_pine_fixed2, model_pine_mixed2, "Pine")

### **Spruce**

In [ ]:
model_spruce_fixed2 = dep_model_all("Spruce","fixed",ingrowth_BA_aprox);

In [ ]:
model_spruce_mixed2 = dep_model_all("Spruce","mixed",ingrowth_BA_aprox);

In [ ]:
plot_trajectories(model_spruce_fixed2, model_spruce_mixed2, "Spruce")

### **Birch**

In [ ]:
model_birch_fixed2 = dep_model_all("Birch","fixed",ingrowth_BA_aprox);

In [ ]:
model_birch_mixed2 = dep_model_all("Birch","mixed",ingrowth_BA_aprox);

In [ ]:
plot_trajectories(model_birch_fixed2, model_birch_mixed2, "Birch")

### **Broadleaf**

In [ ]:
model_broadleaf_fixed2 = dep_model_all("Broadleaf","fixed",ingrowth_BA_aprox);

In [ ]:
model_broadleaf_mixed2 = dep_model_all("Broadleaf","mixed",ingrowth_BA_aprox);

In [ ]:
plot_trajectories(model_broadleaf_fixed2, model_broadleaf_mixed2, "Broadleaf")

## **Inserting cuts from the approximation**

The new idea is to create a model with the non-linear ingrowth function and inserting in it cuts from the respective linear version of the model.

See the [Training a convex model](https://sddp.dev/stable/tutorial/pglib_opf/#Training-a-convex-model) subsection on the $\texttt{SDDP.jl}$ documentation for a good explanation. 

In [ ]:
"""
     inserted_cuts_model(species:String, effect:String)->SDDP.PolicyGraph{Int64}

Returns a non-linear model for a species and effect, with cuts directly extracted from 
the linear version of the model for said species and effect.

# Arguments
- `species::String`: species to be simulated.
- `effect::String`: model for the survival paremeters to be simulated.
"""
function inserted_cuts_model(species, effect)
    model_aprox = dep_model(species, effect, ingrowth_BA_aprox)
    model = dep_model(species, effect, ingrowth_BA)
    SDDP.numerical_stability_report(model_aprox)
    SDDP.numerical_stability_report(model)
    SDDP.train(model_aprox; iteration_limit = 50)
    SDDP.write_cuts_to_file(model_aprox, "cuts/"*species*"_"*effect*".cuts.json")
    SDDP.read_cuts_from_file(model, "cuts/"*species*"_"*effect*".cuts.json")
    return model
end;

### **Pine**

In [ ]:
model_pine_fixed3 = simulate(inserted_cuts_model("Pine", "fixed"));

In [ ]:
model_pine_mixed3 = simulate(inserted_cuts_model("Pine", "mixed"));

In [ ]:
plot_trajectories(model_pine_fixed3, model_pine_mixed3, "Pine")

### **Spruce**

In [ ]:
model_spruce_fixed3 = simulate(inserted_cuts_model("Spruce", "fixed"));

In [ ]:
model_spruce_mixed3 = simulate(inserted_cuts_model("Spruce", "mixed"));

In [ ]:
plot_trajectories(model_spruce_fixed3, model_spruce_mixed3, "Spruce")

### **Birch**

In [ ]:
model_birch_fixed3 = simulate(inserted_cuts_model("Birch", "fixed"));

In [ ]:
model_birch_mixed3 = simulate(inserted_cuts_model("Birch", "mixed"));

In [ ]:
plot_trajectories(model_birch_fixed3, model_birch_mixed3, "Birch")

### **Broadleaf**

In [ ]:
model_broadleaf_fixed3 = simulate(inserted_cuts_model("Other broadleaf species", "fixed"));

In [ ]:
model_broadleaf_mixed3 = simulate(inserted_cuts_model("Other broadleaf species", "mixed"));

In [ ]:
plot_trajectories(model_broadleaf_fixed3, model_broadleaf_mixed3, "Broadleaf")

## **Modification of the *forward* pass**

As the cuts from the linear model are constructed at points from *forward passes* of said model, they may be arbitrarily bad for the non-linear model. Thus, now those *forward passes* are done by the non-linear model, while the *backward passes* still are done by the linear model.

See the [Combining convex and non-convex models](https://sddp.dev/stable/tutorial/pglib_opf/#Combining-convex-and-non-convex-models) subsection on the $\texttt{SDDP.jl}$ documentation for a good explanation. 

In [ ]:
"""
     modified_fwd_model(species:String, effect:String)->SDDP.PolicyGraph{Int64}

Returns a non-linear model for a species and effect, with cuts directly obtained at 
forward passes of said model, while the backward passes come from a linear version 
of the model for said species and effect.

# Arguments
- `species::String`: species to be simulated.
- `effect::String`: model for the survival paremeters to be simulated.
"""
function modified_fwd_model(species, effect)
    model_aprox = dep_model(species, effect, ingrowth_BA_aprox)
    model = dep_model(species, effect, ingrowth_BA)
    SDDP.numerical_stability_report(model_aprox)
    SDDP.numerical_stability_report(model)
    SDDP.train(
        model_aprox;
        forward_pass = SDDP.AlternativeForwardPass(model),
        post_iteration_callback = SDDP.AlternativePostIterationCallback(model),
        iteration_limit = 50,
    )
    return model
end;

### **Pine**

In [ ]:
model_pine_fixed4 = simulate(modified_fwd_model("Pine", "fixed"));

In [ ]:
model_pine_mixed4 = simulate(modified_fwd_model("Pine", "mixed"));

In [ ]:
plot_trajectories(model_pine_fixed4, model_pine_mixed4, "Pine")

### **Spruce**

In [ ]:
model_spruce_fixed4 = simulate(modified_fwd_model("Spruce", "fixed"));

In [ ]:
model_spruce_mixed4 = simulate(modified_fwd_model("Spruce", "mixed"));

In [ ]:
plot_trajectories(model_spruce_fixed4, model_spruce_mixed4, "Spruce")

### **Birch**

In [ ]:
model_birch_fixed4 = simulate(modified_fwd_model("Birch", "fixed"));

In [ ]:
model_birch_mixed4 = simulate(modified_fwd_model("Birch", "mixed"));

In [ ]:
plot_trajectories(model_birch_fixed4, model_birch_mixed4, "Birch")

### **Broadleaf**

In [ ]:
model_broadleaf_fixed4 = simulate(modified_fwd_model("Other broadleaf species", "fixed"));

In [ ]:
model_broadleaf_mixed4 = simulate(modified_fwd_model("Other broadleaf species", "mixed"));

In [ ]:
plot_trajectories(model_broadleaf_fixed4, model_broadleaf_mixed4, "Broadleaf")